
# FairWarn-SHS — Feature-only Baseline Experiment

This notebook trains Logistic Regression, Random Forest, and a feature-only
Multi-Layer Perceptron on the corrected Ghanaian SHS dataset.

The experiment excludes identifiers and the end-of-term aggregate score used
to derive the outcome. All imputation, scaling, and one-hot encoding operations
are fitted inside each training fold to prevent leakage.

**Evaluation protocol:** repeated stratified five-fold cross-validation,
five repeats, producing 25 paired test-fold results per model.


In [ ]:

# Run this first in Google Colab.
!pip -q install pandas numpy scikit-learn matplotlib openpyxl


In [ ]:

from google.colab import files
uploaded = files.upload()
# Upload: FairWarn_SHS_Node_Features.csv


In [ ]:

import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, accuracy_score, brier_score_loss,
    confusion_matrix
)
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

DATA_PATH = "FairWarn_SHS_Node_Features.csv"
nodes = pd.read_csv(DATA_PATH)
labelled = nodes.loc[
    nodes["Label_Available"].eq(1) & nodes["TARGET_AtRisk"].notna()
].copy()
labelled["TARGET_AtRisk"] = labelled["TARGET_AtRisk"].astype(int)

drop_cols = [
    "Node_ID", "Roster_Code", "School_Code", "Class_Code",
    "Label_Available", "TARGET_AtRisk"
]
feature_cols = [c for c in labelled.columns if c not in drop_cols]
X = labelled[feature_cols].copy()
y = labelled["TARGET_AtRisk"].copy()

print("Labelled sample:", len(labelled))
print("Class counts:")
print(y.value_counts().sort_index())
print("At-risk prevalence:", round(y.mean(), 4))


In [ ]:

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_cols)
])

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=3000,
        solver="liblinear", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=500, class_weight="balanced_subsample",
        min_samples_leaf=2, max_features="sqrt",
        random_state=42, n_jobs=-1
    ),
    "Feature-only MLP": MLPClassifier(
        hidden_layer_sizes=(64, 32), activation="relu",
        solver="adam", alpha=0.001, learning_rate_init=0.001,
        max_iter=1000, early_stopping=True,
        validation_fraction=0.15, n_iter_no_change=30,
        random_state=42
    )
}

cv = RepeatedStratifiedKFold(
    n_splits=5, n_repeats=5, random_state=42
)


In [ ]:

rows = []
prediction_rows = []

for model_name, estimator in models.items():
    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", estimator)
        ])

        start = time.perf_counter()
        pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
        train_seconds = time.perf_counter() - start

        start = time.perf_counter()
        proba = pipeline.predict_proba(X.iloc[test_idx])[:, 1]
        inference_seconds = time.perf_counter() - start

        pred = (proba >= 0.5).astype(int)
        y_true = y.iloc[test_idx].to_numpy()

        rows.append({
            "Model": model_name,
            "Fold": fold_id,
            "AUC_ROC": roc_auc_score(y_true, proba),
            "AUC_PR": average_precision_score(y_true, proba),
            "Precision_AtRisk": precision_score(y_true, pred, zero_division=0),
            "Recall_AtRisk": recall_score(y_true, pred, zero_division=0),
            "F1_AtRisk": f1_score(y_true, pred, zero_division=0),
            "Weighted_F1": f1_score(y_true, pred, average="weighted", zero_division=0),
            "Balanced_Accuracy": balanced_accuracy_score(y_true, pred),
            "Accuracy": accuracy_score(y_true, pred),
            "Brier_Score": brier_score_loss(y_true, proba),
            "Train_Seconds": train_seconds,
            "Inference_Seconds": inference_seconds
        })

        for row_idx, true_value, pred_value, probability in zip(
            test_idx, y_true, pred, proba
        ):
            prediction_rows.append({
                "Model": model_name,
                "Fold": fold_id,
                "Node_ID": labelled.iloc[row_idx]["Node_ID"],
                "True_Label": int(true_value),
                "Predicted_Label": int(pred_value),
                "AtRisk_Probability": float(probability)
            })

fold_metrics = pd.DataFrame(rows)
predictions = pd.DataFrame(prediction_rows)
fold_metrics.head()


In [ ]:

metrics = [
    "AUC_ROC", "AUC_PR", "Precision_AtRisk", "Recall_AtRisk",
    "F1_AtRisk", "Weighted_F1", "Balanced_Accuracy",
    "Accuracy", "Brier_Score", "Train_Seconds", "Inference_Seconds"
]

summary_rows = []
for model_name, grp in fold_metrics.groupby("Model", sort=False):
    row = {"Model": model_name}
    for metric in metrics:
        row[f"{metric}_Mean"] = grp[metric].mean()
        row[f"{metric}_SD"] = grp[metric].std(ddof=1)
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary


In [ ]:

# Stability plot
plt.figure(figsize=(8, 5))
data = [
    fold_metrics.loc[fold_metrics["Model"].eq(name), "AUC_PR"].to_numpy()
    for name in models
]
plt.boxplot(data, tick_labels=list(models.keys()), showmeans=True)
plt.ylabel("AUC-PR")
plt.title("Baseline AUC-PR across 25 repeated test folds")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


In [ ]:

fold_metrics.to_csv("baseline_fold_metrics.csv", index=False)
summary.to_csv("baseline_summary_mean_sd.csv", index=False)
predictions.to_csv("baseline_fold_predictions.csv", index=False)

files.download("baseline_summary_mean_sd.csv")
files.download("baseline_fold_metrics.csv")
files.download("baseline_fold_predictions.csv")
